## Init stuff

In [ ]:
import os

if os.getcwd() == '/home/dcor/niskhizov':
    os.chdir('//home/dcor/niskhizov/PhysicalAdverserialProj/')
    on_remote = True
else:
    on_remote = False
    

In [ ]:
from comet_ml import start, ExistingExperiment
from comet_ml.integration.pytorch import log_model

experiment = start(
  api_key="Bg5eubpUjdi2CCiiA5OSoltfw",
  project_name="physicaladvproj",
  workspace="dannynis"
)

# Get the experiment ID, end it, and re-attach to ensure code capture
experiment_key = experiment.get_key()
print(f"Experiment key: {experiment_key}")


In [ ]:
%matplotlib inline

In [ ]:

import torch
import torch.nn as nn
import torchvision.models as models

import glob
import tqdm
import matplotlib.pyplot as plt
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
import cv2
import time
import time
from tqdm import tqdm
import copy
from IPython.display import display, Image, clear_output
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import datetime


import cv2
import os
import glob


import kornia
import tqdm

import torchvision

tt = torchvision.transforms.ToTensor()

import cv2
import cv2.aruco as aruco
import numpy as np

from consts import border_size, displayed_aruco_code, marker_size, latent_size, latent_batch_size

# from classfier import *
# from classfier_clip import *
# from classfier_dino import *
from classfier_ensemble import predict_raw, orig_clases, model, weights
from classfier_ensemble import predict_raw as predict_raw_dev
from classfier_ensemble import predict_raw as predict_raw_test
from functools import partial

from classfier_test import model as model_test
from classfier import model as model_dev

model_name = model if type(model) == str else model.__class__.__name__

print(f'For training using classfier_dev model: {model.__class__.__name__}  ')
print('###############################')
print(f'For Dev using classfier_dev model: {model_dev.__class__.__name__}  ')
print('###############################')
print(f'For Test using classfier_test model: {model_test.__class__.__name__}  ')




if 'weight_dict' in predict_raw.__code__.co_varnames:

    classfiers_weights_dict = {
    'inception': 0.25,
    'resnet': 0.25,
    'vgg': 0.25,
    'vit': 0.25,
    'dino': 0.0}
    
    experiment.log_parameters(classfiers_weights_dict)
    predict_raw = partial(predict_raw, weights_dict=classfiers_weights_dict)
# predict_raw_per_model = partial(predict_raw_per_model, weights_dict=classfiers_weights_dict)




import pickle as pkl


# orig_clases = torch.tensor([999,700, 419]).cuda() # apple
# orig_clases = torch.tensor([948,956]).cuda()

print( '#################################')
print( f'ORIG CLASES {orig_clases}')
photometric_calibrations_dir = './photometric_calibrations'
# load latest photometric calibration by time
photometric_calibration_files = glob.glob(os.path.join(photometric_calibrations_dir, 'photometric_calibration_*.pkl'))
photometric_calibration_files.sort(key=os.path.getmtime, reverse=True)
photometric_calibration_path = photometric_calibration_files[0]
print(f'############## Loading photometric calibration data from {photometric_calibration_path} ##############')

with open(photometric_calibration_path, "rb") as f:
    data = pkl.load(f)

height = data['height']
width = data['width']

resizer = torchvision.transforms.Resize((height, width))
device = 'cuda' if torch.cuda.is_available() else 'cpu'


# Define parameters for ArUco marker detection
aruco_dict_type = cv2.aruco.DICT_4X4_50 # Change dictionary type if needed
marker_length = 0.05  # Marker length in meters (adjust as needed)
aruco_dict = cv2.aruco.getPredefinedDictionary(aruco_dict_type)

marker_id = displayed_aruco_code
marker_size = marker_size  # Size in pixels
marker_image = cv2.aruco.generateImageMarker(aruco_dict, marker_id, marker_size)


aruco_dict = aruco.getPredefinedDictionary(aruco_dict_type)
parameters = aruco.DetectorParameters()

# Detect ArUco markers
detector = aruco.ArucoDetector(aruco_dict, parameters)


from diffusers import StableDiffusionPipeline
import torch

# Load stable diffusion model


def decode_latents_grad(latents):
    # latents = F.interpolate(latents, (64, 64), mode='bilinear', align_corners=False)
    latents = 1 / 0.18215 * latents

    imgs = vae.decode(latents).sample

    imgs = (imgs / 2 + 0.5).clamp(0, 1)

    return imgs

def decode_latents(latents):
    # latents = F.interpolate(latents, (64, 64), mode='bilinear', align_corners=False)
    with torch.no_grad():
        with torch.amp.autocast(device):
            latents = 1 / 0.18215 * latents

            with torch.no_grad():
                imgs = vae.decode(latents).sample

            imgs = (imgs / 2 + 0.5).clamp(0, 1)

    return imgs

def encode_imgs(imgs):
    # imgs: [B, 3, H, W]
    with torch.no_grad():
        with torch.amp.autocast(device):
            imgs = 2 * imgs - 1

            posterior = vae.encode(imgs).latent_dist
            latents = posterior.sample() * 0.18215

    return latents



class framesDataset(Dataset):
    def __init__(self, frames, Hs):
        self.frames = frames
        self.Hs = Hs

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, idx):
        frame = self.frames[idx]
        H = self.Hs[idx]

        # Convert to tensor and normalize
        frame_tensor = tt(frame)

        return frame_tensor, H.astype(np.float32)
    





def warp(decoded_latents,H_t):
    dst_img_shape = valid_frames[0].shape[:2]
    warped_imgs = []
    for decoded_latent in decoded_latents:
        img = decoded_latent.unsqueeze(0).float().repeat(H_t.shape[0], 1, 1, 1)
        w=  kornia.geometry.transform.warp_perspective(img, H_t, dst_img_shape)
        warped_imgs.append(w)
    return torch.stack(warped_imgs, dim=0)#.squeeze(1)


vae = None
valid_frames = None

os.makedirs('./results', exist_ok=True)
curr_without_sec = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
curr_without_sec = curr_without_sec.replace(" ", "_").replace(":", "_")




pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4")

vae =  pipe.vae.to(device).eval()
# compile vae for faster execution
# vae = torch.compile(pipe.vae.to(device).eval())


try:
    first_class_idx = orig_clases[0].item()
    first_class_name = weights.meta['categories'][first_class_idx]
    experiment_name = f"{model_name}_{curr_without_sec}_{first_class_name}"
    experiment.set_name(experiment_name)
    print(f"Experiment name set to: {experiment_name}")
except Exception as e:
    print(f"Could not set experiment name: {e}")

In [ ]:
from consts import border_size, displayed_aruco_code, latent_size, latent_batch_size
print(f'Using border size {border_size} for ArUco detection')
print(f'Using latent size {latent_size}x{latent_size}')
print(f'Using latent batch size {latent_batch_size}')
# border_glow_margin = 5

In [ ]:
def find_border_drop_point(gray, c):
    sub = np.subtract
    add = np.add
    borders_drop_points = []
    for idx,operators in enumerate(([sub,sub],[add,sub],[add,add],[sub,add])):

        margin = 1
        a,b = int(c[idx][0]), int(c[idx][1])
        diag_idxs = np.arange(5)

        nca = operators[0](a, diag_idxs)
        ncb = operators[1](b, diag_idxs)
        nc = np.stack([nca, ncb], axis=1)

        diag_line_vals = gray[nc[:, 1], nc[:, 0]].astype(np.float32)
        diag_line_vals_diff = np.diff(diag_line_vals)
        if np.all(diag_line_vals_diff >= 0):
            # did not find border drop point, use the first point
            borders_drop_points.append((nca[0], ncb[0]))
            continue
        diag_line_vals_diff_first_neg = min(np.where(diag_line_vals_diff < 0)[0][0] + margin, len(diag_line_vals_diff)-1)
        new_a = nca[diag_line_vals_diff_first_neg]
        new_b = ncb[diag_line_vals_diff_first_neg]
        borders_drop_points.append((new_a, new_b))

        
    return np.array(borders_drop_points)


caps_dir = 'captures_frames_multiview'
ls = os.listdir(f'./{caps_dir}')
captures = [f for f in ls if f.startswith('captures_frames_multiview_') ]
captures = sorted(captures, key=lambda x: int(x.split('_')[-1]))
cap_dir = f'./{caps_dir}/{captures[-1]}'#f'captures_frames_multiview_{len(captures)-1}'

# cap_dir = f'./{caps_dir}/captures_frames_multiview_64'#{captures[-1]}'#f'captures_frames_multiview_{len(captures)-1}'
# # cap_dir = f'./{caps_dir}/captures_frames_multiview_{len(captures)-1}'
# # cap_dir = f'./{caps_dir}/captures_frames_multiview_52'#{captures[-1]}'#f'captures_frames_multiview_{len(captures)-1}'


print('using capture dir', cap_dir)


valid_frame_paths = glob.glob(f'{cap_dir}/*.png')

valid_frames = []

Hs = []

# orig_img_corners = np.array([[0,0],[l_size_w,0],[l_size_w,l_size_h],[0,l_size_h]], dtype=np.float32)
orig_img_corners = np.array([[border_size,border_size],[width-border_size,border_size],[width-border_size,height-border_size],[border_size,height-border_size]], dtype=np.float32)


found_aruco_count = 0
outlier_clases = []
for path in tqdm.tqdm_notebook(valid_frame_paths):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    corners, ids, _ = detector.detectMarkers(gray)

    if ids is None:
        continue
    
    found_aruco_count += 1

    with torch.no_grad():
        # pr = resnet_predict_raw(tt(img).cuda().unsqueeze(0))
        # print(model)
        pr = predict_raw(tt(img).cuda().unsqueeze(0))

        pres_img = img.copy()
        if ids is not None:
            aruco.drawDetectedMarkers(pres_img, corners, ids)
        cat = weights.meta["categories"][pr.argmax(1)]
        if not on_remote:
            cv2.putText(pres_img, f'pred: {cat} {pr.max().item():.2f}', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
            cv2.imshow('pres', cv2.cvtColor(pres_img,cv2.COLOR_RGB2BGR))
            cv2.waitKey(1)
        

    if ids is not None and displayed_aruco_code in ids and pr.argmax(1) in orig_clases:
        
        displayed_aruco_code_index = np.where(ids.flatten() == displayed_aruco_code)[0][0]
        corners = corners[displayed_aruco_code_index][0]
        
        unbordred_corners = find_border_drop_point(gray, corners)

        # unbordred_corners = np.array([[c[0][0]-per_corner_glow_margin, c[0][1]-border_glow_margin],
        #                     [c[1][0]+border_glow_margin, c[1][1]-border_glow_margin],
        #                     [c[2][0]+border_glow_margin, c[2][1]+border_glow_margin],
        #                     [c[3][0]-border_glow_margin, c[3][1]+border_glow_margin]])

        # dst_pts = unbordred_corners
        H, _ = cv2.findHomography(orig_img_corners, unbordred_corners, cv2.RANSAC)

        Hs.append(H)
        valid_frames.append(img)
        

    else :
        outlier_clases.append(pr.argmax(1))
cv2.destroyAllWindows()
print(f"Detected ArUco markers in {found_aruco_count} out of {len(valid_frame_paths)} frames.")
print(f"Found {len(valid_frames)} valid frames with ArUco markers and original classes.")

# print('DEBUG training')
# valid_frames = [valid_frames[0]]*10
# Hs = [Hs[0]]*10

# Use ALL valid frames instead of limiting to 10
# print(f"Using ALL {len(valid_frames)} frames for training.")

random_idx = np.random.permutation(min(len(valid_frames), 5000))
valid_frames = [valid_frames[i] for i in random_idx]
Hs = [Hs[i] for i in random_idx]

print(f"Total valid frames to be used for training: {len(valid_frames)}")

# No need to subset - use all valid frames
# valid_frames and Hs already contain all valid frames


ds = framesDataset(valid_frames, Hs)


print('creating dataloader')
# Calculate split sizes that sum to dataset length
total_len = len(ds)
train_len = int(total_len * 0.8)
val_len = int(total_len * 0.1)
test_len = total_len - train_len - val_len  # Ensure exact sum

print(f"Dataset split: train={train_len}, val={val_len}, test={test_len}, total={total_len}")
train,val,test = torch.utils.data.random_split(ds, [train_len, val_len, test_len])
train_loader = DataLoader(train, batch_size=1, shuffle=True)#, num_workers=2, persistent_workers=True)
val_loader = DataLoader(val, batch_size=len(val), shuffle=False)#), num_workers=2, persistent_workers=True)
test_loader = DataLoader(test, batch_size=len(test), shuffle=False)#, num_workers=2, persistent_workers=True)

print('dataloader created')
    
# latent.requires_grad = True



def random_blur(img):
    k_size = np.random.choice([3,5,7])
    sigma = np.random.uniform(0.1, 1.0)
    return T.GaussianBlur(kernel_size=k_size, sigma=(sigma, sigma))(img)

# Define augmentation functions with Gaussian blur to simulate camera/projector defocus
jitter = T.Compose([
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    random_blur
])

jitter_total_photo = T.Compose([
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5))  # Slight blur for unfocused camera/projector
])

jitter_with_hue = T.Compose([
    T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
    T.GaussianBlur(kernel_size=5, sigma=(0.1, 1.0))  # More aggressive blur
])

augmentor_model = data['augmentor'].to(device).eval()
# augmentor_model = torch.compile(augmentor_model)
# mapper = lambda x: jitter(x)

In [ ]:
categories = weights.meta['categories']

In [ ]:
categories[414]

In [ ]:
categories.index('paper towel')

In [ ]:
torch.stack(outlier_clases).unique(return_counts=True)

In [ ]:
# No need to load single frame - we'll iterate through all frames
print(f"Dataset contains {len(train_loader)} training batches")
print(f"Total training frames: {len(train_loader.dataset)}")
print(f"Validation frames: {len(val_loader.dataset) if len(val_loader.dataset) > 0 else 0}")
print(f"Test frames: {len(test_loader.dataset) if len(test_loader.dataset) > 0 else 0}")
print("Ready to process ALL frames during training!")

In [ ]:
sample_frame, sample_H = next(iter(train_loader))


In [ ]:
# Display a sample frame from the dataset
sample_frame, sample_H = next(iter(train_loader))
plt.imshow(sample_frame[0].permute(1,2,0).cpu().numpy())
plt.title("Sample training frame")
plt.axis('off')
plt.show()

patch = torch.zeros((1, 3, height, width), device=device).to(device)

# patch = torch.zeros((1, 3, int(height * 1.5) , int(width * 1.5) ), device=device).to(device)
H = sample_H.to(device)
frame = sample_frame.to(device)
w_mask = warp(patch * 0 + 1, H)
w_patch = warp(patch, H)

# Full replacement for maximum effect
blend_ratio = 1.0
blended_frames = ((w_mask != 0) * -blend_ratio + 1) * frame + w_patch * blend_ratio

plt.imshow(blended_frames[0,0,...].permute(1,2,0).cpu().numpy())
plt.title("Blended frame with patch")
plt.axis('off')
plt.show()

In [ ]:
# Load successful latent if available, otherwise reinitialize
print("Attempting to load successful latent or reinitializing...")

# Try to load a previously successful latent
# try:
#     latent_batch = torch.load('./results/successful_latent_2025-09-20_18_23.pt').to(device)
#     print("✓ Loaded successful latent from previous run!")
# except:
    # If no successful latent available, use a different initialization strategy
# latent_batch = torch.randn((10, 4, latent_size, latent_size), device=device) * 0.8  # Larger initial noise
latent_batch = torch.randn((latent_batch_size, 4, latent_size, latent_size), device=device) * 0.8  # Larger initial noise
# latent_batch = torch.load('./results/full_latent_batch_epoch_33_2025-11-14_03_10.pt')
# print('!!! Loaded latent batch from file ./results/full_latent_batch_epoch_33_2025-11-14_03_10.pt !!!')
# print("Using fresh random initialization with larger variance")

latent_batch.requires_grad = True

# Use a more conservative optimizer with higher learning rate
latent_opt = torch.optim.Adam([latent_batch], lr=0.1)  # Higher LR, simpler optimizer
# add cosine annealing scheduler  after epoch 3


orig_clases_np = orig_clases.cpu().numpy()
print(f"Latent shape: {latent_batch.shape}")
print(f"Original classes to avoid: {orig_clases_np}")
# print(f"Other classes available: {len(total_clases_without_orig)}")

# Test initial prediction
with torch.no_grad():
    test_patch = resizer(decode_latents(latent_batch).float())
    print(f"Generated patch shape: {test_patch.shape}")
    print("Initialization complete!")

In [ ]:

aug_weight = 0.4
# Training parameters
num_epochs = 300  # Increased since augmentation training is more challenging
blend_ratio = 1.0  # Full replacement
best_loss = float('inf')
best_latent = None
best_success_rate = 0
success_count = 0
training_stopped = False  # Flag for early stopping when patch > 50%
aug_rate = None
to_rejuvenate = False
use_scheduler = False  # Set to False to disable learning rate scheduling

# Scheduler configuration
scheduler_config = {
    'type': 'OneCycleLR',  # Options: 'OneCycleLR', 'StepLR', 'CosineAnnealingLR', None
    'max_lr': 0.1,
    'pct_start': 0.0,  # For OneCycleLR
    'step_size': 50,   # For StepLR
    'gamma': 0.9,      # For StepLR
    'T_max': num_epochs,  # For CosineAnnealingLR
}

def create_scheduler(optimizer, config, total_steps):
    """Create a learning rate scheduler based on configuration.
    
    Args:
        optimizer: The optimizer to schedule
        config: Dictionary with scheduler configuration
        total_steps: Total number of training steps/epochs
        
    Returns:
        scheduler or None if disabled
    """
    if config is None or config.get('type') is None:
        print("📉 No scheduler - using constant learning rate")
        return None
    
    scheduler_type = config['type']
    
    if scheduler_type == 'OneCycleLR':
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=config.get('max_lr', 0.1),
            total_steps=total_steps,
            pct_start=config.get('pct_start', 0.0)
        )
        print(f"📈 Initialized OneCycleLR with max_lr={config.get('max_lr', 0.1)}, pct_start={config.get('pct_start', 0.0):.4f}")
    
    elif scheduler_type == 'StepLR':
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=config.get('step_size', 50),
            gamma=config.get('gamma', 0.9)
        )
        print(f"📈 Initialized StepLR with step_size={config.get('step_size', 50)}, gamma={config.get('gamma', 0.9)}")
    
    elif scheduler_type == 'CosineAnnealingLR':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=config.get('T_max', total_steps),
            eta_min=config.get('eta_min', 0.0001)
        )
        print(f"📈 Initialized CosineAnnealingLR with T_max={config.get('T_max', total_steps)}")
    
    else:
        print(f"⚠️ Unknown scheduler type: {scheduler_type}, using no scheduler")
        return None
    
    return scheduler

# Initialize scheduler (or None if disabled)
if use_scheduler:
    scheduler = create_scheduler(latent_opt, scheduler_config, num_epochs)
else:
    scheduler = None
    print("📉 Scheduler disabled - using constant learning rate")

# Helper to get current learning rate
def get_current_lr(optimizer, scheduler):
    if scheduler is not None:
        return scheduler.get_last_lr()[0]
    return optimizer.param_groups[0]['lr']

experiment.log_parameters({
    "aug_weight": aug_weight,
    "num_epochs": num_epochs,
    "blend_ratio": blend_ratio,
    "initial_lr": 0.1,
    "latent_batch_size": latent_batch.shape[0],
    "latent_size": latent_size,
    "model_name": model.__class__.__name__,
    "model_dev_name": model_dev.__class__.__name__,
    "model_test_name": model_test.__class__.__name__,
    "scheduler": scheduler_config.get('type') if use_scheduler else "None",
    "pct_start": scheduler_config.get('pct_start', 0.0),
    "to_rejuvenate": to_rejuvenate,
    "use_scheduler": use_scheduler
})

In [ ]:
training_stopped = False

## Trainig

In [ ]:
# Robust adversarial attack working on ALL frames with augmentation training
print("Starting ROBUST adversarial attack on ALL frames in dataset...")
print("🔧 Augmentations enabled: jitter on patch, augmentor on patch, jitter_total_photo on final image")
print("This will make the patch robust against real-world variations!")

# Loss tracking
losses = []
success_rates = []

# Per-patch performance tracking
num_patches = latent_batch.shape[0]

# Better target class selection - classes that are visually very different from vehicles
target_classes = torch.tensor([
#     # 1,    # tench (fish)
#     # 2,    # goldfish  
#     # 3,    # great white shark
#     # 4,    # tiger shark
#     # 50,   # American robin (bird)
#     # 285,  # Egyptian cat
#     # 281,  # tabby cat
#     # 805,  # soccer ball
#     # 999,  # toilet tissue
#     # 732,  # Polaroid camera (this worked before!)
    # 530,  # digital clock
#     # 575, # golf cart
#     # 627, # limousine
    
    
], device=device)
# target_classes = torch.tensor([x for x in range(1000) if x not in orig_clases.cpu().numpy()], device=device)
# target_classes = torch.tensor([], device=device)

print(f"Target classes: {[weights.meta['categories'][i] for i in target_classes.cpu().numpy()]}")
print(f"Training on {len(train_loader)} batches per epoch (ALL frames)")


augmentor = lambda x: augmentor_model(x).to(device) * aug_weight + x * (1-aug_weight)


In [ ]:
weights.meta['categories'].index('tennis ball')

In [ ]:
for x in weights.meta['categories']:
    if 'golf' in x:
        print(x)

In [ ]:
patch_history = []

In [ ]:
best_patch_idx = None
epoch = 0

In [ ]:
experiment.end()
print("Re-attaching to experiment...")
experiment = ExistingExperiment(
  api_key="Bg5eubpUjdi2CCiiA5OSoltfw",
  previous_experiment=experiment_key
)

In [ ]:
## training loop
while epoch < num_epochs:
    epoch_losses = []
    epoch_success_rates = []

    patch_success_history = {i: [] for i in range(num_patches)}  # Track each patch over time
    patch_augmented_success_history = {i: [] for i in range(num_patches)}  # Track augmented performance


    # Process ALL frames in the training set
    for batch_idx, (frames_batch, H_t_batch) in tqdm.tqdm(enumerate(train_loader)):

        latent_opt.zero_grad()
        
        frames_batch = frames_batch.to(device)
        H_t_batch = H_t_batch.to(device)
        
        # Generate adversarial patch from latent
        adv_patch = resizer(decode_latents_grad(latent_batch).float())

        if len(patch_history) < 10000:
            patch_history.append(adv_patch.detach().cpu())
        
        # Apply jitter augmentation to the patch for robustness
        if torch.rand(1).item() > 0.3:  # 70% chance to apply patch augmentation
            adv_patch_aug = jitter(adv_patch)
        else:
            adv_patch_aug = adv_patch
            
        # Apply augmentor transformation to patch for additional robustness
        # Skip augmentor for now due to size constraints - use only jitter augmentations
        if torch.rand(1).item() > 0.3:  # 70% chance to apply augmentor
            adv_patch_aug = torch.stack([augmentor(x).to(device) for x in adv_patch_aug])
        
        # Apply warping to current batch of frames
        w_mask = warp(adv_patch_aug * 0 + 1, H_t_batch)
        w_patch = warp(adv_patch_aug, H_t_batch)
        
        # Full replacement for maximum effect
        blended_frames = ((w_mask != 0) * -blend_ratio + 1) * frames_batch + w_patch * blend_ratio
        blended_frames = blended_frames.squeeze(1)
        # Apply jitter_total_photo to the entire blended result for robustness
        if torch.rand(1).item() > 0.3:  # 70% chance to apply total photo jitter
            blended_frames = jitter_total_photo(blended_frames)
        
        # Reshape for batch processing
        batch_frames = blended_frames.view(-1, *blended_frames.shape[1:])
        
        # Preprocess for model
        # processed_batch = preprocess(batch_frames)
        
        # Get model predictions
        # logits = model(processed_batch)
        with torch.autocast(device_type=device):
            logits = predict_raw(batch_frames)
            if (logits != logits).any():
                raise
            probs = torch.softmax(logits, dim=1)


        # Improved loss calculation
        # 1. Strong penalty for original classes
        orig_class_probs = probs[:, orig_clases]
        orig_loss = 5.0 * torch.log(orig_class_probs.sum(dim=1) + 1e-10).mean()
        
        # 2. Strong reward for our specific target classes
        if target_classes.numel() > 0:
            target_probs = probs[:, target_classes]
            target_loss = -3.0 * torch.log(target_probs.max(dim=1)[0] + 1e-10).mean()
        else:
            target_loss = 0
        
        # 3. Minimal regularization
        # patch_reg = 0.0001 * (adv_patch - 0.5).pow(2).mean()
        
        # Combined loss
        # print(target_loss)
        total_loss = orig_loss + target_loss #+ patch_reg
        
        # Backward pass
        total_loss.backward()
        if (latent_batch != latent_batch).any():
            raise
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_([latent_batch], max_norm=3.0)
        
        latent_opt.step()
        
        # Track metrics for this batch
        epoch_losses.append(total_loss.item())
        
        # Evaluation for this batch
        with torch.no_grad():
            predictions = logits.argmax(dim=1)
            successful_attacks = sum(pred.item() not in orig_clases_np for pred in predictions)
            success_rate = successful_attacks / len(predictions)
            epoch_success_rates.append(success_rate)

            # Log batch metrics
            step = epoch * len(train_loader) + batch_idx
            experiment.log_metric("total_loss", total_loss.item(), step=step)
            experiment.log_metric("orig_loss", orig_loss.item(), step=step)
            if isinstance(target_loss, torch.Tensor):
                experiment.log_metric("target_loss", target_loss.item(), step=step)
            else:
                experiment.log_metric("target_loss", target_loss, step=step)
            experiment.log_metric("batch_success_rate", success_rate, step=step)
    


    # Update scheduler once per epoch (if enabled)
    if scheduler is not None:
        scheduler.step() 
    
    # Calculate epoch averages
    avg_epoch_loss = np.mean(epoch_losses)
    avg_epoch_success = np.percentile(epoch_success_rates, 90) #np.mean(epoch_success_rates)
    
    losses.append(avg_epoch_loss)
    success_rates.append(avg_epoch_success)

    # Log epoch metrics
    experiment.log_metric("epoch_avg_loss", avg_epoch_loss, step=epoch)
    experiment.log_metric("epoch_avg_success", avg_epoch_success, step=epoch)
    current_lr = get_current_lr(latent_opt, scheduler)
    experiment.log_metric("learning_rate", current_lr, step=epoch)
    experiment.log_metric("aug_weight", aug_weight, step=epoch)


    # if epoch % 5:
    #     masked_probs = probs.clone().detach()
    #     masked_probs[:, orig_clases] = -1  # Invalidate original classes
    #     target_classes = torch.cat([target_classes, masked_probs.sort(dim=1, descending=True)[1][:,:5].flatten().unique()]).unique().long()
    #     print(f"Refined target classes at epoch: {[weights.meta['categories'][int(i)] for i in target_classes.cpu().numpy()]}")
        
    # Per-patch performance evaluation on augmented data (every 5 epochs or when successful)
    if epoch % 5 == 0 or avg_epoch_success > 0.3:
        print(f"\n📊 Evaluating individual patch performance on augmented data...")
        
        with torch.no_grad():
            # Generate all patches
            all_patches = resizer(decode_latents(latent_batch).float())
            
            # Test each patch individually on a small subset of validation data
            val_batch_limit = min(3, len(val_loader))  # Use first 3 validation batches
            
            for patch_idx in range(num_patches):
                patch_clean_successes = 0
                patch_aug_successes = 0
                total_tests_clean = 0
                total_tests_aug = 0
                
                single_patch = all_patches[patch_idx:patch_idx+1]  # Keep batch dimension
                
                for val_batch_idx, (val_frames, val_H_t) in enumerate(val_loader):
                    if val_batch_idx >= val_batch_limit:
                        break
                        
                    val_frames = val_frames.to(device)
                    val_H_t = val_H_t.to(device)
                    
                    # === CLEAN PATCH TEST ===
                    w_mask = warp(single_patch * 0 + 1, val_H_t)
                    w_patch = warp(single_patch, val_H_t)
                    clean_blended = ((w_mask != 0) * -blend_ratio + 1) * val_frames + w_patch * blend_ratio
                    clean_batch = clean_blended.view(-1, *clean_blended.shape[2:])
                    
                    # clean_processed = preprocess(clean_batch)
                    # clean_logits = model(clean_processed)
                    clean_logits = predict_raw_dev(clean_batch)
                    clean_predictions = clean_logits.argmax(dim=1)
                    
                    # === AUGMENTED PATCH TEST ===
                    aug_patch = jitter(single_patch)
                    aug_patch = torch.stack([augmentor(x).to(device) for x in aug_patch])

                    w_mask_aug = warp(aug_patch * 0 + 1, val_H_t)
                    w_patch_aug = warp(aug_patch, val_H_t)
                    aug_blended = ((w_mask_aug != 0) * -blend_ratio + 1) * val_frames + w_patch_aug * blend_ratio
                    aug_blended = aug_blended.squeeze(0)
                    aug_blended = jitter_total_photo(aug_blended)
                    aug_batch = aug_blended.view(-1, *aug_blended.shape[1:])
                    
                    # aug_processed = preprocess(aug_batch)
                    # aug_logits = model(aug_processed)
                    aug_logits = predict_raw_dev(aug_batch)
                    aug_predictions = aug_logits.argmax(dim=1)
                    
                    # Count successes
                    for pred in clean_predictions:
                        if pred.item() not in orig_clases_np:
                            patch_clean_successes += 1
                        total_tests_clean += 1
                    
                    for pred in aug_predictions:
                        if pred.item() not in orig_clases_np:
                            patch_aug_successes += 1
                        total_tests_aug += 1
                
                ######### Visualize some augmented predictions for debugging #########
                if epoch % 1 == 0 and best_patch_idx is not None and patch_idx == best_patch_idx:
                    for i in range(min(3, len(aug_batch))):
                        img = ((aug_batch[i].permute(1,2,0).cpu().numpy()) * 255).astype(np.uint8)
                        img = np.ascontiguousarray(img)
                        prob =100* torch.softmax(aug_logits[i], dim=0)[aug_predictions[i]]
                        res = categories[aug_predictions[i]]
                        cv2.putText(img, f'Pred: {res}: {prob:.2f}%', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)


                        if epoch % 10 == 0 :
                            # plt.title(f"Val Batch {val_batch_idx} - Augmented Patch {i+1}")
                            plt.imshow(img)

                            plt.axis('off')
                            plt.show()
                            print(f"  Predicted: {res} (Prob: {prob})")
                        
                        # Log image to Comet
                experiment.log_image(img, name=f"Augmented Patch idx {patch_idx+1}", step=epoch)
                    
                    # Log the patch itself to Comet
                patch_img = (single_patch[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
                patch_img = np.ascontiguousarray(patch_img)
                experiment.log_image(patch_img, name=f"Patch_{patch_idx+1}", step=epoch)
                
                # Also log the augmented patch
                aug_patch_img = (aug_patch[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
                aug_patch_img = np.ascontiguousarray(aug_patch_img)
                experiment.log_image(aug_patch_img, name=f"Patch_{patch_idx+1}_Augmented", step=epoch)
                #####################################################################
                # Calculate success rates for this patch
                clean_rate = patch_clean_successes / total_tests_clean if total_tests_clean > 0 else 0
                aug_rate = patch_aug_successes / total_tests_aug if total_tests_aug > 0 else 0
                if aug_rate == 1.0 and aug_weight > 0.9 :
                    print('Found patch with 100% augmented success rate at full augmentation weight!')
                # Store in history
                patch_success_history[patch_idx].append(clean_rate)
                patch_augmented_success_history[patch_idx].append(aug_rate)
        

        
        # Report top performing patches
        if len(patch_success_history[0]) > 0:  # If we have data
            print(f"🏆 TOP 5 PATCHES (Augmented Performance):")
            
            # Get latest augmented performance for each patch
            latest_aug_performance = [(i, patch_augmented_success_history[i][-1]) for i in range(num_patches)]
            latest_aug_performance.sort(key=lambda x: x[1], reverse=True)
            
            # Check if any patch achieved > 50% success rate
            best_patch_idx, best_patch_rate = latest_aug_performance[0]

            if torch.mean(torch.tensor([x[1] for x in latest_aug_performance[:1]])) > 0.7:    
                print(f"  🎯 HIGH SUCCESS RATE ACHIEVED!")
                if aug_weight < 1:
                    if aug_weight < 0.7:
                        aug_weight += 0.1
                    elif aug_weight < 0.9:
                        aug_weight += 0.05
                    else:
                        aug_weight += 0.01
                    aug_weight = min(aug_weight, 1.0)  # Cap at 1.0
                    print(f"Augmentation weight increased to {aug_weight}")
                    augmentor = lambda x: augmentor_model(x).to(device) * aug_weight + x * (1-aug_weight)
            
            for rank, (patch_idx, aug_rate) in enumerate(latest_aug_performance[:5], 1):
                clean_rate = patch_success_history[patch_idx][-1]
                robustness = (aug_rate / clean_rate) if clean_rate > 0 else 0
                print(f"  {rank}. Patch #{patch_idx+1:2d}: Clean {clean_rate:.1%} | Aug {aug_rate:.1%} | Robust {robustness:.1%}")
                
                # Log top patch metrics
                experiment.log_metric(f"patch_{patch_idx}_clean_rate", clean_rate, step=epoch)
                experiment.log_metric(f"patch_{patch_idx}_aug_rate", aug_rate, step=epoch)
                experiment.log_metric(f"patch_{patch_idx}_robustness", robustness, step=epoch)
            
            # Log best patch (rank 1) success rates for tracking plots
            best_patch_idx_eval, best_patch_aug_rate_eval = latest_aug_performance[0]
            best_patch_clean_rate_eval = patch_success_history[best_patch_idx_eval][-1]
            experiment.log_metric("best_patch_clean_rate_eval", best_patch_clean_rate_eval, step=epoch)
            experiment.log_metric("best_patch_aug_rate_eval", best_patch_aug_rate_eval, step=epoch)
            
            # Log worst patch (last rank) success rates for tracking plots
            worst_patch_idx_eval, worst_patch_aug_rate_eval = latest_aug_performance[-1]
            worst_patch_clean_rate_eval = patch_success_history[worst_patch_idx_eval][-1]
            experiment.log_metric("worst_patch_clean_rate_eval", worst_patch_clean_rate_eval, step=epoch)
            experiment.log_metric("worst_patch_aug_rate_eval", worst_patch_aug_rate_eval, step=epoch)
            
            # Show worst performers too
            print(f"📉 BOTTOM 3 PATCHES (Augmented Performance):")
            for rank, (patch_idx, aug_rate) in enumerate(latest_aug_performance[-3:], 1):
                clean_rate = patch_success_history[patch_idx][-1]
                print(f"  {rank}. Patch #{patch_idx+1:2d}: Clean {clean_rate:.1%} | Aug {aug_rate:.1%}")
            # Check for 90% threshold on individual patches
            if aug_weight>=0.5 and best_patch_rate > 0.6:

                
                # Save the best latent
                best_patch_latent = latent_batch[best_patch_idx:best_patch_idx+1].clone().detach()
                torch.save(best_patch_latent, f'./results/successful_patch_{best_patch_idx+1}_epoch_{epoch}_{model_name}_{curr_without_sec}.pt')
                experiment.log_asset(f'./results/successful_patch_{best_patch_idx+1}_epoch_{epoch}_{model_name}_{curr_without_sec}.pt')
                
                # Save the full latent batch for completeness
                torch.save(latent_batch.clone().detach(), f'./results/full_latent_batch_epoch_{epoch}_{model_name}_{curr_without_sec}.pt')
                experiment.log_asset(f'./results/full_latent_batch_epoch_{epoch}_{model_name}_{curr_without_sec}.pt')
                
                print(f"💾 Saved successful patch to: ./results/successful_patch_{best_patch_idx+1}_epoch_{epoch}_{model_name}_{curr_without_sec}.pt")
                print(f"💾 Saved full batch to: ./results/full_latent_batch_epoch_{epoch}_{model_name}_{curr_without_sec}.pt")
                
                # Set flags to break out of training
                if aug_weight >= 1 and best_patch_rate >= 0.9:
                    print(f"\n🎉 BREAKTHROUGH! Patch #{best_patch_idx+1} achieved {best_patch_rate:.1%} success rate!")
                    print(f"🛑 STOPPING TRAINING - 90% threshold exceeded!")
                    training_stopped = True
                    break
            # trim latent to top 5 patches
            # if [x[1] for x in latest_aug_performance[:5]][0] > 0.2:
            #     print(f"🔧 Focusing training on top 5 patches based on augmented performance...")
            #     latent_batch = latent_batch[[x[0] for x in latest_aug_performance[:5]]].clone().detach()
            #     latent_batch.requires_grad = True
            #     latent_opt = torch.optim.Adam([latent_batch], lr=0.1)
            #     scheduler = torch.optim.lr_scheduler.StepLR(latent_opt, step_size=50, gamma=0.9)
            #     num_patches = latent_batch.shape[0]
            #     print(f"🔧 Trimmed latent batch to top {num_patches} patches for focused training")
            if to_rejuvenate and [x[1] for x in latest_aug_performance[:5]][0] > 0.2:
                print(f"🔧 Rejuvenating weakest patches based on augmented performance...")
                # Save scheduler state to persist the schedule (if scheduler exists)
                scheduler_state = scheduler.state_dict() if scheduler is not None else None
                
                latent_batch = latent_batch.clone().detach()
                latent_batch_best = latent_batch[[x[0] for x in latest_aug_performance[:5]]]
                latent_batch[[x[0] for x in latest_aug_performance[-5:]]] = latent_batch_best + (torch.randn_like(latent_batch_best) * 0.1)
                latent_batch.requires_grad = True
                
                # Re-create optimizer for the new latent batch
                latent_opt = torch.optim.Adam([latent_batch], lr=0.1)
                # Re-initialize scheduler if it was enabled
                if scheduler is not None and use_scheduler:
                    scheduler = create_scheduler(latent_opt, scheduler_config, num_epochs)
                    if scheduler_state is not None:
                        scheduler.load_state_dict(scheduler_state)
                        print(f"🔧 Scheduler state restored (Step: {scheduler.last_epoch}/{num_epochs})")
                
                num_patches = latent_batch.shape[0]
                print(f"🔧 Rejuvenated latent batch weakest 5 patches with noise from top 5 performers")
        print()  # Add spacing
    
    # Check if training was stopped due to high-performing patch
    if training_stopped:
        print(f"Training stopped early due to patch achieving success rate")
        break
        
    if avg_epoch_success > 0:
        success_count += 1
        
    # Save best model based on success rate
    if avg_epoch_success > best_success_rate or (avg_epoch_success == best_success_rate and avg_epoch_loss < best_loss):
        best_success_rate = avg_epoch_success
        best_loss = avg_epoch_loss
        best_latent = latent_batch.clone().detach()
        
    # Progress reporting
    if epoch % 1 == 0 or avg_epoch_success > 0.5:
        current_lr = get_current_lr(latent_opt, scheduler)
        
        print(f"Epoch {epoch:3d}/{num_epochs} | "
              f"Loss: {avg_epoch_loss:.3f} | "
              f"Success: {avg_epoch_success:.1%} | "
              f"Frames: {len(train_loader)} | "
              f"LR: {current_lr:.5f}")
        
    experiment.log_metric("best_success_rate", best_success_rate)
    experiment.log_metric("best_loss", best_loss)


    
    # Early stopping if consistently successful across all frames
    # if len(success_rates) >= 10 and np.mean(success_rates[-10:]) >= 0.8:
    #     print(f"\n🎉 EXCELLENT! Consistent success across ALL frames at epoch {epoch}")
    #     print(f"Recent 10-epoch average: {np.mean(success_rates[-10:]):.1%}")
    #     break
        
    # Adaptive learning rate boost for successful attacks
    # if avg_epoch_success > 0.5 and epoch < 150:
    #     for param_group in latent_opt.param_groups:
    #         param_group['lr'] = min(param_group['lr'] * 1.01, 0.15)
    
    epoch += 1    

print(f"\n🏁 Optimization complete!")
print(f"Best success rate: {best_success_rate:.1%}")
print(f"Best loss: {best_loss:.3f}")



if best_latent is not None:
    torch.save(best_latent, f"best_latent_final_{model_name}.pt")
    experiment.log_asset(f"best_latent_final_{model_name}.pt")

In [ ]:
aug_weight

## Exploration

In [ ]:
    print(f"\n📊 Evaluating individual patch performance on augmented data...")
    categories = weights.meta["categories"]
    with torch.no_grad():
        # Generate all patches
        all_patches = resizer(decode_latents(latent_batch).float())
        
        # Test each patch individually on a small subset of validation data
        val_batch_limit = min(3, len(val_loader))  # Use first 3 validation batches
        
        for patch_idx in range(num_patches):
            patch_clean_successes = 0
            patch_aug_successes = 0
            total_tests = 0
            
            single_patch = all_patches[patch_idx:patch_idx+1]  # Keep batch dimension
            
            for val_batch_idx, (val_frames, val_H_t) in enumerate(val_loader):
                if val_batch_idx >= val_batch_limit:
                    break
                    
                val_frames = val_frames.to(device)
                val_H_t = val_H_t.to(device)
                
                # === CLEAN PATCH TEST ===
                w_mask = warp(single_patch * 0 + 1, val_H_t)
                w_patch = warp(single_patch, val_H_t)
                clean_blended = ((w_mask != 0) * -blend_ratio + 1) * val_frames + w_patch * blend_ratio
                clean_batch = clean_blended.view(-1, *clean_blended.shape[2:])
                
                # clean_processed = preprocess(clean_batch)
                # clean_logits = model(clean_processed)
                clean_logits = predict_raw_test(clean_batch)
                clean_predictions = clean_logits.argmax(dim=1)
                
                # === AUGMENTED PATCH TEST ===
                aug_patch = jitter(single_patch)
                try:
                    aug_patch = torch.stack([augmentor(x).to(device) for x in aug_patch])
                except:
                    pass  # Use only jitter if augmentor fails
                
                w_mask_aug = warp(aug_patch * 0 + 1, val_H_t)
                w_patch_aug = warp(aug_patch, val_H_t)
                aug_blended = ((w_mask_aug != 0) * -blend_ratio + 1) * val_frames + w_patch_aug * blend_ratio
                aug_blended = aug_blended.squeeze(0)
                aug_blended = jitter_total_photo(aug_blended)
                aug_batch = aug_blended.view(-1, *aug_blended.shape[1:])
                
                
                # aug_processed = preprocess(aug_batch)
                # aug_logits = model(aug_processed)
                aug_logits = predict_raw(aug_batch)
                aug_predictions = aug_logits.argmax(dim=1)


                
                # Count successes
                for pred in clean_predictions:
                    if pred.item() not in orig_clases_np:
                        patch_clean_successes += 1
                    total_tests += 1
                
                for pred in aug_predictions:
                    if pred.item() not in orig_clases_np:
                        patch_aug_successes += 1
            
            # Calculate success rates for this patch
            clean_rate = patch_clean_successes / total_tests if total_tests > 0 else 0
            aug_rate = patch_aug_successes / total_tests if total_tests > 0 else 0



            # if aug_rate > 0.9:
            #     raise
            # Store in history
            patch_success_history[patch_idx].append(clean_rate)
            patch_augmented_success_history[patch_idx].append(aug_rate)
    
    # Report top performing patches
    if len(patch_success_history[0]) > 0:  # If we have data
        print(f"🏆 TOP 5 PATCHES (Augmented Performance):")
        
        # Get latest augmented performance for each patch
        latest_aug_performance = [(i, patch_augmented_success_history[i][-1]) for i in range(num_patches)]
        latest_aug_performance.sort(key=lambda x: x[1], reverse=True)
        
        # Check if any patch achieved > 50% success rate
        best_patch_idx, best_patch_rate = latest_aug_performance[0]
        
        for rank, (patch_idx, aug_rate) in enumerate(latest_aug_performance[:5], 1):
            clean_rate = patch_success_history[patch_idx][-1]
            robustness = (aug_rate / clean_rate) if clean_rate > 0 else 0
            print(f"  {rank}. Patch #{patch_idx+1:2d}: Clean {clean_rate:.1%} | Aug {aug_rate:.1%} | Robust {robustness:.1%}")
        
        # Show worst performers too
        print(f"📉 BOTTOM 3 PATCHES (Augmented Performance):")
        for rank, (patch_idx, aug_rate) in enumerate(latest_aug_performance[-3:], 1):
            clean_rate = patch_success_history[patch_idx][-1]
            print(f"  {rank}. Patch #{patch_idx+1:2d}: Clean {clean_rate:.1%} | Aug {aug_rate:.1%}")
        
        # Check for 50% threshold on individual patches
        if best_patch_rate > 0.5:
            print(f"\n🎉 BREAKTHROUGH! Patch #{best_patch_idx+1} achieved {best_patch_rate:.1%} success rate!")
            print(f"🛑 STOPPING TRAINING - 50% threshold exceeded!")
            
            # Save the successful patch immediately
            current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H_%M")
            
            # Save the best latent
            best_patch_latent = latent_batch[best_patch_idx:best_patch_idx+1].clone().detach()
            torch.save(best_patch_latent, f'./results/successful_patch_{best_patch_idx+1}_epoch_{epoch}_{current_time}.pt')
            
            # Save the full latent batch for completeness
            torch.save(latent_batch.clone().detach(), f'./results/full_latent_batch_epoch_{epoch}_{current_time}.pt')
            
            print(f"💾 Saved successful patch to: ./results/successful_patch_{best_patch_idx+1}_epoch_{epoch}_{current_time}.pt")
            print(f"💾 Saved full batch to: ./results/full_latent_batch_epoch_{epoch}_{current_time}.pt")
            
            # Set flags to break out of training
            training_stopped = True
            
    
    print()  # Add spacing

In [ ]:
with torch.no_grad():
    single_patch = all_patches[best_patch_idx:best_patch_idx+1]  # Keep batch dimension

    for val_batch_idx, (val_frames, val_H_t) in enumerate(val_loader):
        if val_batch_idx >= val_batch_limit:
            break
                
        val_frames = val_frames.to(device)
        val_H_t = val_H_t.to(device)

        # === CLEAN PATCH TEST ===
        w_mask = warp(single_patch * 0 + 1, val_H_t)
        w_patch = warp(single_patch, val_H_t)
        clean_blended = ((w_mask != 0) * -blend_ratio + 1) * val_frames + w_patch * blend_ratio
        clean_batch = clean_blended.view(-1, *clean_blended.shape[2:])

        # clean_processed = preprocess(clean_batch)
        # clean_logits = model(clean_processed)
        clean_logits = predict_raw_test(clean_batch)
        clean_predictions = clean_logits.argmax(dim=1)

        # === AUGMENTED PATCH TEST ===
        aug_patch = jitter(single_patch)
        try:
            aug_patch = torch.stack([augmentor(x).to(device) for x in aug_patch])
        except:
            pass  # Use only jitter if augmentor fails

        w_mask_aug = warp(aug_patch * 0 + 1, val_H_t)
        w_patch_aug = warp(aug_patch, val_H_t)
        aug_blended = ((w_mask_aug != 0) * -blend_ratio + 1) * val_frames + w_patch_aug * blend_ratio
        aug_blended = aug_blended.squeeze(0)
        aug_blended = jitter_total_photo(aug_blended)
        aug_batch = aug_blended.view(-1, *aug_blended.shape[1:])
        aug_logits = predict_raw_test(aug_batch)
        aug_predictions = aug_logits.argmax(dim=1)
        for i in range(min(5, len(aug_batch))):
            img = ((aug_batch[i].permute(1,2,0).cpu().numpy()) * 255).astype(np.uint8)
            img = np.ascontiguousarray(img)
            prob =100* torch.softmax(aug_logits[i], dim=0)[aug_predictions[i]]
            res = categories[aug_predictions[i]]

            # plt.title(f"Val Batch {val_batch_idx} - Augmented Patch {i+1}")
            cv2.putText(img, f'Pred: {res}: {prob:.2f}%', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)
            plt.imshow(img)

            plt.axis('off')
            plt.show()
            print(f"  Predicted: {res} (Prob: {prob})")

In [ ]:
# Present the best performing patch    
print("=== PRESENTING THE BEST PERFORMING PATCH ===")

if 'latent_batch' in locals():
    with torch.no_grad():
        # Generate all patches
        all_patches = resizer(decode_latents(latent_batch).float())
        
        # Find the best performing patch if we have tracking data
        if 'patch_augmented_success_history' in locals() and len(patch_augmented_success_history[0]) > 0:
            # Get final performance for each patch
            final_aug_performance = [(i, patch_augmented_success_history[i][-1]) for i in range(num_patches)]
            final_aug_performance.sort(key=lambda x: x[1], reverse=True)
            
            best_patch_idx, best_aug_rate = final_aug_performance[0]
            best_clean_rate = patch_success_history[best_patch_idx][-1]
            
            print(f"🏆 BEST PERFORMING PATCH: #{best_patch_idx+1}")
            print(f"   Clean Success Rate: {best_clean_rate:.1%}")
            print(f"   Augmented Success Rate: {best_aug_rate:.1%}")
            print(f"   Robustness Score: {(best_aug_rate/best_clean_rate):.2f}" if best_clean_rate > 0 else "   Robustness Score: N/A")
            
            # Display the best patch
            best_patch = all_patches[best_patch_idx]
            
            plt.figure(figsize=(12, 8))
            
            # Plot 1: Best patch visualization
            plt.subplot(2, 3, 1)
            plt.imshow(best_patch.permute(1,2,0).cpu().numpy())
            plt.title(f'Best Patch #{best_patch_idx+1}\nAug Success: {best_aug_rate:.1%}')
            plt.axis('off')
            
            # Plot 2: Performance comparison with other top patches
            plt.subplot(2, 3, 2)
            top_5_indices = [x[0] for x in final_aug_performance[:5]]
            top_5_aug_rates = [x[1] for x in final_aug_performance[:5]]
            top_5_clean_rates = [patch_success_history[i][-1] for i in top_5_indices]
            
            x = range(len(top_5_indices))
            width = 0.35
            plt.bar([i - width/2 for i in x], top_5_clean_rates, width, label='Clean', alpha=0.8)
            plt.bar([i + width/2 for i in x], top_5_aug_rates, width, label='Augmented', alpha=0.8)
            plt.xlabel('Top 5 Patches')
            plt.ylabel('Success Rate')
            plt.title('Top 5 Patch Performance')
            plt.xticks(x, [f'#{i+1}' for i in top_5_indices])
            plt.legend()
            
            # Plot 3: Show second and third best patches for comparison
            if len(final_aug_performance) >= 2:
                plt.subplot(2, 3, 4)
                second_best_idx = final_aug_performance[1][0]
                second_patch = all_patches[second_best_idx]
                plt.imshow(second_patch.permute(1,2,0).cpu().numpy())
                plt.title(f'2nd Best: Patch #{second_best_idx+1}\nAug: {final_aug_performance[1][1]:.1%}')
                plt.axis('off')
            
            if len(final_aug_performance) >= 3:
                plt.subplot(2, 3, 5)
                third_best_idx = final_aug_performance[2][0]
                third_patch = all_patches[third_best_idx]
                plt.imshow(third_patch.permute(1,2,0).cpu().numpy())
                plt.title(f'3rd Best: Patch #{third_best_idx+1}\nAug: {final_aug_performance[2][1]:.1%}')
                plt.axis('off')
            
            # Plot 4: Performance evolution of best patch
            if len(patch_augmented_success_history[best_patch_idx]) > 1:
                plt.subplot(2, 3, 3)
                evaluation_points = list(range(len(patch_augmented_success_history[best_patch_idx])))
                plt.plot(evaluation_points, patch_success_history[best_patch_idx], 'o-', label='Clean', linewidth=2)
                plt.plot(evaluation_points, patch_augmented_success_history[best_patch_idx], 's-', label='Augmented', linewidth=2)
                plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='50% Threshold')
                plt.xlabel('Evaluation Point')
                plt.ylabel('Success Rate')
                plt.title(f'Patch #{best_patch_idx+1} Evolution')
                plt.legend()
                plt.grid(True, alpha=0.3)
            
            # Plot 5: Worst patch for comparison
            if len(final_aug_performance) > 0:
                plt.subplot(2, 3, 6)
                worst_patch_idx = final_aug_performance[-1][0]
                worst_patch = all_patches[worst_patch_idx]
                plt.imshow(worst_patch.permute(1,2,0).cpu().numpy())
                plt.title(f'Worst: Patch #{worst_patch_idx+1}\nAug: {final_aug_performance[-1][1]:.1%}')
                plt.axis('off')
            
            plt.tight_layout()
            plt.show()
            
            # Print detailed statistics
            print(f"\n📊 DETAILED STATISTICS FOR BEST PATCH #{best_patch_idx+1} ({latent_batch.shape[-2]}x{latent_batch.shape[-1]}):")
            print(f"   Shape: {best_patch.shape}")
            print(f"   Pixel value range: [{best_patch.min():.3f}, {best_patch.max():.3f}]")
            print(f"   Mean pixel value: {best_patch.mean():.3f}")
            print(f"   Standard deviation: {best_patch.std():.3f}")
            
            # Save the best patch as an image
            import torchvision.transforms as transforms
            current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H_%M")
            patch_path = f'./results/best_patch_{model_name}_{latent_batch.shape[-2]}x{latent_batch.shape[-1]}_{best_patch_idx+1}_{current_time}.png'
            
            # Convert to PIL and save
            to_pil = transforms.ToPILImage()
            patch_pil = to_pil(best_patch.cpu())
            patch_pil.save(patch_path)
            print(f"💾 Best patch saved as image: {patch_path}")
            
            if best_aug_rate > 0.5:
                print(f"\n🎉 CONGRATULATIONS! Best patch achieved {best_aug_rate:.1%} success rate!")
                print(f"   This patch successfully fools the classifier on {best_aug_rate:.1%} of augmented test cases!")
            elif best_aug_rate > 0.3:
                print(f"\n🎯 GOOD PERFORMANCE! Best patch achieved {best_aug_rate:.1%} success rate!")
                print(f"   Consider training longer or adjusting parameters for even better results.")
            else:
                print(f"\n💡 MODERATE PERFORMANCE: Best patch achieved {best_aug_rate:.1%} success rate.")
                print(f"   Try different initialization, learning rates, or augmentation strategies.")
        
        else:
            # If no tracking data, just show all patches
            print("No performance tracking data available. Showing all generated patches:")
            
            # Calculate grid size
            n_patches = all_patches.shape[0]
            cols = min(5, n_patches)
            rows = (n_patches + cols - 1) // cols
            
            plt.figure(figsize=(3*cols, 3*rows))
            for i in range(n_patches):
                plt.subplot(rows, cols, i+1)
                plt.imshow(all_patches[i].permute(1,2,0).cpu().numpy())
                plt.title(f'Patch #{i+1}')
                plt.axis('off')
            
            plt.tight_layout()
            plt.show()
            
            print(f"Generated {n_patches} patches total")

else:
    print("❌ No patches available. Please run the training cell first.")

In [ ]:
save_dir = f'./results/{model_name}_{curr_without_sec}_top_patches/'
os.makedirs(save_dir, exist_ok=True)
for patch_idx, rate in final_aug_performance:
    if rate > 0.2:
        cur_patch = all_patches[patch_idx]
        patch_pil = to_pil(cur_patch.cpu())
        patch_path = save_dir + f'{patch_idx}_{str(rate)[:4]}.png'
        patch_pil.save(patch_path)

In [ ]:
save_dir

In [ ]:
model_name

In [ ]:
# end comet experiment
experiment.end()


In [ ]:
raise

In [ ]:
raise

## pres

In [ ]:
best_patch.shape

In [ ]:
# augmentor = lambda x: x.to(device)#augmentor_model(x).to(device)

best_patch = cv2.imread('./results/best_patch_6_2025-11-13_13_20.png')
best_patch = tt(cv2.cvtColor(best_patch, cv2.COLOR_BGR2RGB))

best_patch_aug = augmentor(best_patch)

to_pil = transforms.ToPILImage()


In [ ]:
all_blended = []
for valid_frame,hom in zip(valid_frames,Hs):
    valid_frame = tt(valid_frame)
    h= torch.tensor(hom).unsqueeze(0).cuda().float()
    w_mask = warp(best_patch_aug * 0 + 1, h).cpu()[:,0,0,:,:]
    w_patch = warp(best_patch_aug, h).cpu()[:,0,0,:,:]
    blended = ((w_mask != 0) * -blend_ratio + 1) * valid_frame + w_patch * blend_ratio

    all_blended.append(blended)

all_blended = torch.stack(all_blended)

In [ ]:
with torch.no_grad():
    pr_all = predict_raw(all_blended.cuda())

In [ ]:
for b,pr in zip(all_blended,pr_all):
    pres_img = cv2.cvtColor(np.array(to_pil(b)),cv2.COLOR_RGB2BGR)
    cv2.putText(pres_img, f'pred: {cat} {pr.max().item():.2f}', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)
    cv2.imshow('pres',pres_img)
    cv2.waitKey(1)
    time.sleep(0.1)

In [ ]:
plt.imshow(blended.cpu().permute(1,2,0).numpy())

In [ ]:
# create gif from patch history
print("Creating GIF from patch history...")
import imageio
gif_images = []
gif_blended_frames = []
gif_blended_frames_aug = []
for idx, patch_img in enumerate(patch_history[::2]):  # Sample every 10th for brevity
    chosen_patch = patch_img[3].cuda().unsqueeze(0)  # Choose patch index 3 for consistency
    img = (chosen_patch.permute(0,2,3,1).squeeze(0).cpu().numpy() * 255).astype(np.uint8)
    img = np.ascontiguousarray(img)

    cv2.putText(img, f'Patch Evolution {idx}', (1,3), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)
    gif_images.append(img)

    w_mask = warp(chosen_patch * 0 + 1, H_t_batch)
    w_patch = warp(chosen_patch, H_t_batch)

    # Full replacement for maximum effect
    blended_frames = ((w_mask != 0) * -blend_ratio + 1) * frames_batch + w_patch * blend_ratio
    blended_frames = blended_frames.squeeze(1)
    with torch.no_grad():
        logits = predict_raw(blended_frames.view(-1, *blended_frames.shape[1:]))
        probs = torch.softmax(logits, dim=1)
        pred = logits.argmax(dim=1)[0]
        prob =100* probs[0,pred]

    res = categories[pred]
    blended_img = (blended_frames[0].permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
    blended_img = np.ascontiguousarray(blended_img)

    # place text under the patch at the bottom of the image
    cv2.putText(blended_img, f'Step {idx} Pred: {res}: {prob:.2f}%', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)

    gif_blended_frames.append(blended_img)

    # chosen_patch_aug = jitter(chosen_patch)
    chosen_patch_aug = torch.stack([augmentor(x).to(device) for x in chosen_patch])
    w_mask_aug = warp(chosen_patch_aug * 0 + 1, H_t_batch)
    w_patch_aug = warp(chosen_patch_aug, H_t_batch)
    blended_frames_aug = ((w_mask_aug != 0) * -blend_ratio + 1) * frames_batch + w_patch_aug * blend_ratio
    blended_frames_aug = blended_frames_aug.squeeze(1)
    with torch.no_grad():
        logits = predict_raw(blended_frames_aug.view(-1, *blended_frames_aug.shape[1:]))
        probs = torch.softmax(logits, dim=1)
        pred = logits.argmax(dim=1)[0]
        prob =100* probs[0,pred]
    res = categories[pred]
    blended_img_aug = (blended_frames_aug[0].permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
    blended_img_aug = np.ascontiguousarray(blended_img_aug)
    cv2.putText(blended_img_aug, f'Step {idx} Pred: {res}: {prob:.2f}%', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)
    gif_blended_frames_aug.append(blended_img_aug)

imageio.mimsave(f'./results/patch_evolution_blended_augmented_{curr_without_sec}.gif', gif_blended_frames_aug, fps=5)
imageio.mimsave(f'./results/patch_evolution_blended_{curr_without_sec}.gif', gif_blended_frames, fps=5)
imageio.mimsave(f'./results/patch_evolution_{curr_without_sec}.gif', gif_images, fps=5)
print("GIF saved!")

In [ ]:
video_name = './results/patch_evolution.avi'
height, width, layers =gif_images[0].shape
video = cv2.VideoWriter(video_name, 0, 15, (width,height))
for captured in gif_images:
    captured = cv2.cvtColor(captured, cv2.COLOR_RGB2BGR)
    video.write(captured)

In [ ]:
plt.imshow(gif_blended_frames_aug[-1])

In [ ]:
plt.imshow(blended_frames_aug[0].cpu().permute(1,2,0).numpy())

In [ ]:
plt.imshow(w_patch_aug[0][0].cpu().squeeze(0).permute(1,2,0).numpy())

In [ ]:
plt.imshow(chosen_patch_aug.permute(0,2,3,1).squeeze(0).cpu().numpy())

In [ ]:
w_patch_aug.shape

In [ ]:
  
        # with torch.no_grad():
        #     # Generate all patches
        #     all_patches = resizer(decode_latents(latent_batch).float())
            
        #     # Test each patch individually on a small subset of validation data
        #     val_batch_limit = min(3, len(val_loader))  # Use first 3 validation batches
            
        #     for patch_idx in range(num_patches):
        #         patch_clean_successes = 0
        #         patch_aug_successes = 0
        #         total_tests_clean = 0
        #         total_tests_aug = 0
                
        #         single_patch = all_patches[patch_idx:patch_idx+1]  # Keep batch dimension
                
        #         for val_batch_idx, (val_frames, val_H_t) in enumerate(val_loader):
        #             if val_batch_idx >= val_batch_limit:
        #                 break
                        
        #             val_frames = val_frames.to(device)
        #             val_H_t = val_H_t.to(device)
                    
        #             # === CLEAN PATCH TEST ===
        #             w_mask = warp(single_patch * 0 + 1, val_H_t)
        #             w_patch = warp(single_patch, val_H_t)
        #             clean_blended = ((w_mask != 0) * -blend_ratio + 1) * val_frames + w_patch * blend_ratio
        #             clean_batch = clean_blended.view(-1, *clean_blended.shape[2:])
                    
        #             # clean_processed = preprocess(clean_batch)
        #             # clean_logits = model(clean_processed)
        #             clean_logits = predict_raw(clean_batch)
        #             clean_predictions = clean_logits.argmax(dim=1)
                    
        #             # === AUGMENTED PATCH TEST ===
        #             aug_patch = jitter(single_patch)
        #             aug_patch = torch.stack([augmentor(x).to(device) for x in aug_patch])

        #             w_mask_aug = warp(aug_patch * 0 + 1, val_H_t)
        #             w_patch_aug = warp(aug_patch, val_H_t)
        #             aug_blended = ((w_mask_aug != 0) * -blend_ratio + 1) * val_frames + w_patch_aug * blend_ratio
        #             aug_blended = aug_blended.squeeze(0)
        #             aug_blended = jitter_total_photo(aug_blended)
        #             aug_batch = aug_blended.view(-1, *aug_blended.shape[1:])
                    
        #             # aug_processed = preprocess(aug_batch)
        #             # aug_logits = model(aug_processed)
        #             aug_logits = predict_raw(aug_batch)
        #             aug_predictions = aug_logits.argmax(dim=1)
                    
        #             # Count successes
        #             for pred in clean_predictions:
        #                 if pred.item() not in orig_clases.cpu().numpy():
        #                     patch_clean_successes += 1
        #                 total_tests_clean += 1
                    
        #             for pred in aug_predictions:
        #                 if pred.item() not in orig_clases.cpu().numpy():
        #                     patch_aug_successes += 1
        #                 total_tests_aug += 1
                
        #         # Calculate success rates for this patch
        #         clean_rate = patch_clean_successes / total_tests_clean if total_tests_clean > 0 else 0
        #         aug_rate = patch_aug_successes / total_tests_aug if total_tests_aug > 0 else 0
        #         if aug_rate == 1.0 and aug_weight > 0.9 :
        #             print('Found patch with 100% augmented success rate at full augmentation weight!')
        #         # Store in history
        #         patch_success_history[patch_idx].append(clean_rate)
        #         patch_augmented_success_history[patch_idx].append(aug_rate)
        

In [ ]:
# plt.imshow(val_frames[4].detach().cpu().permute(1,2,0).numpy())
# plt.axis('off')


In [ ]:
# plt.imshow(0.6 * aug_blended[4].detach().cpu().permute(1,2,0).numpy())
# plt.axis('off')
